In [ ]:
import pandas as pd
import numpy as np
from scipy.stats import false_discovery_control
#from bokeh.models import FixedTicker, FuncTickFormatter 
from scipy.stats import chi2
from plot_tools import *
#from bokeh.io import output_notebook

from snp_analysis_tools_sherlock import *
from scipy.stats import binom
import iqplot
import bokeh.plotting
import bokeh.io
import holoviews as hv
from holoviews import dim, opts
import bokeh.models
from bokeh.layouts import gridplot
from glob import glob

hv.extension('bokeh')

In [ ]:
def subsample_and_plot(good_freq, good_depth, color = 'grey',  x_limits = (-.5, 7.5), alpha = 1.):
    good_freq_subsampled = good_freq.copy()
    good_depth_subsampled = good_depth.copy()
    if len(good_freq) > 1000:
        good_freq_subsampled, good_depth_subsampled= get_plotting_snps(good_freq, good_depth)
    tidy_data_freq = get_tidy_df(good_freq_subsampled)
    tidy_data_depth = get_tidy_df(good_depth_subsampled, value_name = 'depth')
    tidy_data_freq_good = tidy_data_freq.loc[~np.isnan(tidy_data_depth['depth']), :]
    snps_plot = make_mesocosm_timecourse(tidy_data_freq_good.sort_values('passage'),
                                                    color = color,
                                        alpha = alpha,
                                                     limits = x_limits)
    return snps_plot
    
def get_tidy_df(filtered_freq, e003_metadata, value_name = 'freq'):
    filtered_freq.index.name = 'site_id'
    tidy_data = filtered_freq.reset_index().melt(id_vars = ['site_id'], var_name = 'sample', value_name = value_name)

    tidy_data = tidy_data.loc[tidy_data['sample'].isin(e003_metadata.index.values)]

    tidy_data['passage'] = tidy_data["sample"].transform(lambda x: e003_metadata.loc[x, 'passage'])
    tidy_data['passage'] = pd.to_numeric(tidy_data['passage'])

    tidy_data['mesocosm'] = tidy_data["sample"].transform(lambda x: e003_metadata.loc[x, 'mesocosm'])
    
  #  tidy_data['inoculumn'] = tidy_data["sample"].transform(lambda x: e003_metadata.loc[x, 'inoculumn'])
   # tidy_data['inoculumn_sample'] = tidy_data['inoculumn'].transform(get_in)
    
    
    return tidy_data

In [ ]:
def make_mesocosm_timecourse(tidy_data, title = '',
                            color = 'grey', alpha = 1.,
                                              limits = (-.5,7.5)   ):
    hv_curve = hv.Curve(data = tidy_data.sort_values(by='passage'),
                kdims=['passage', ],
                vdims=['freq','site_id']
                ).groupby('site_id'
                ).opts(width=500, height=250,
                color = color,
                ylabel = 'Strain AA Allele Frequency',
                       xlabel='Passage',
                title = title,
                #show_grid=True,
                line_width = 1.,
                       alpha = alpha,
               xlim = limits,
                ylim = (-0.05, 1.03)).overlay()
    
    return hv_curve

In [ ]:
df1 = pd.read_csv('~/git/coalescence-pilot-mgx/workflow/out/102478/to_save_dist1.csv.gz').set_index('site_id')
df1.head()

In [ ]:
good = df1.isna().sum(axis=1)
good = good[good==0].index.values
good

In [ ]:
e003_metadata = pd.read_csv('e003_coalescence_metadata_round4.csv').set_index('sample')
df1 = pd.read_csv('~/git/coalescence-pilot-mgx/workflow/out/midas2_output/to_save_dist1.csv.gz').set_index('site_id')
#df1 = pd.read_csv('~/git/coalescence-pilot-mgx/workflow/out/midas2_output/to_save_dist2.csv.gz').set_index('site_id')
#df1 = df1[['A_C12_C5_AF_AE_mBHI_mBHI_4_S36', 'A_C4_C5_AF_AE_mBHI_mBHI_2_S28',
 #      'C5-e003Coalescence-mBHI-inoculumn-redo',
  #      'C5-e003Coalescence-mBHI-p5',
   #    'C5-e003Coalescence-mBHI-p7', 'C_C4_C5_AF_AE_mBHI_mBHI_6_S220']]
print(len(df1))
good = df1.isna().sum(axis=1)
good = good[good==0].index.values
d1=df1.loc[good,:]
random_inds = np.random.choice(df1.index.values, 1000)
tidy=get_tidy_df(df1.loc[random_inds,:], e003_metadata, value_name = 'freq')
p1 = make_mesocosm_timecourse(tidy,color=bokeh.palettes.Set2[8][2],alpha = .1)
p1

In [ ]:
df1.columns.values

In [ ]:
#df1 = pd.read_csv('~/git/coalescence-pilot-mgx/workflow/out/1013462/to_save_dist2.csv.gz').set_index('site_id')
df2 = pd.read_csv('~/git/coalescence-pilot-mgx/workflow/out/midas2_output/to_save_dist2.csv.gz').set_index('site_id')
#df1 = df1[['A_C12_C5_AF_AE_mBHI_mBHI_4_S36', 'A_C4_C5_AF_AE_mBHI_mBHI_2_S28',
 #      'C5-e003Coalescence-mBHI-inoculumn-redo',
  #      'C5-e003Coalescence-mBHI-p5',
   #    'C5-e003Coalescence-mBHI-p7', 'C_C4_C5_AF_AE_mBHI_mBHI_6_S220']]
print(len(df2))
good = df2.isna().sum(axis=1)
good = good[good==0].index.values
d1=df2.loc[good,:]
random_inds = np.random.choice(df2.index.values, 1000)
tidy=get_tidy_df(df2.loc[random_inds,:], e003_metadata, value_name = 'freq')
p2 = make_mesocosm_timecourse(tidy,color=bokeh.palettes.Set2[8][-1],alpha = .1)
p2


In [ ]:
from plot_tools import *
sp = 102506
ino = 'AA-AF-mGAM'
fname = f'/Users/sophiewalton/git/coalescence-pilot-mgx/workflow/report/track_snpsv2_ALL_bootstrapv3/{sp}/{ino}_parent1_info.csv'
df_both = get_both_dfs(fname)
e003_metadata = pd.read_csv('e003_coalescence_metadata_round4_good.csv').set_index('sample')
df_both = df_both.loc[np.intersect1d(df_both.index.values,e003_metadata.index.values),:]
df_both=df_both.loc[df_both['total_shift']<.1,:]

df_both_meta = pd.concat([df_both,e003_metadata.loc[np.intersect1d(df_both.index.values,
                                                                                   e003_metadata.index.values),:]],
                                         axis=1).reset_index()
df_both_meta_e = df_both_meta.loc[df_both_meta['sample'].isin(df1.columns.values),:]
df_both_meta_e

In [ ]:
p=hv.render(p2*p1)
tick_font_size ='15px'
label_font_size = '22px'

p.yaxis.axis_label_text_font_style = 'normal'
p.xaxis.axis_label_text_font_style = 'normal'
p.xaxis.axis_label_text_font_size=label_font_size
p.xaxis.major_label_text_font_size=tick_font_size
p.yaxis.axis_label_text_font_size=label_font_size
p.yaxis.major_label_text_font_size=tick_font_size
p.yaxis.axis_label = 'Non Ref Allele Freq'
p.xaxis.axis_label = 'Timepoint'
p.xaxis.minor_tick_line_color= None
p.yaxis.minor_tick_line_color= None
df_both_meta_e = df_both_meta_e.sort_values(by='passage')
p.line(df_both_meta_e['passage'].values, df_both_meta_e['boot_med1'].values, color = 'black',
       alpha =1., line_width = 5)
bokeh.io.show(p)

In [ ]:
p.output_backend='svg'
export_plot_pdf(p,'snps_ecoli')

In [ ]:
from plot_tools import *
sp = 102506
ino = 'AA-AF-mGAM'
fname = f'/Users/sophiewalton/git/coalescence-pilot-mgx/workflow/report/track_snpsv2_ALL_bootstrapv3/{sp}/{ino}_parent1_info.csv'
df_both = get_both_dfs(fname)
e003_metadata = pd.read_csv('e003_coalescence_metadata_round4_good.csv').set_index('sample')
df_both = df_both.loc[np.intersect1d(df_both.index.values,e003_metadata.index.values),:]
df_both=df_both.loc[df_both['total_shift']<.1,:]

df_both_meta = pd.concat([df_both,e003_metadata.loc[np.intersect1d(df_both.index.values,
                                                                                   e003_metadata.index.values),:]],
                                         axis=1).reset_index()
df_both_meta_e = df_both_meta.loc[df_both_meta['sample'].isin(df1.columns.values),:]
df_both_meta_e

In [ ]:
df_both_meta['mesocosm'].unique()

In [ ]:
p3 = hv.render((p2*p1*p4).opts(width = 700))
tick_font_size ='15px'
label_font_size = '22px'

p3.xaxis.axis_label_text_font_size=label_font_size
p3.xaxis.axis_label='Timepoint'
p3.yaxis.axis_label= 'Non-ref Allele Freq'
p3.xaxis.major_label_text_font_size=tick_font_size
p3.yaxis.axis_label_text_font_size=label_font_size
p3.yaxis.major_label_text_font_size=tick_font_size
p3.x_range = bokeh.models.Range1d(-.5,7.5)
p3.xaxis.minor_tick_line_color= None
p3.yaxis.minor_tick_line_color= None

In [ ]:
bokeh.io.show(p3)

In [ ]:
df1 = pd.read_csv('~/git/coalescence-pilot-mgx/workflow/out/102478/to_save_dist1.csv.gz').set_index('site_id')
print(len(df1))
good = df1.isna().sum(axis=1)
good = good[good==0].index.values
d1=df1.loc[good,:]
random_inds = np.random.choice(df1.index.values, 1000)
tidy=get_tidy_df(df1.loc[random_inds,:], e003_metadata, value_name = 'freq')
tidy_med = tidy.groupby(['sample']).median(numeric_only=True).reset_index()
tidy_med['site_id']='-'
p3 = make_mesocosm_timecourse(tidy_med,
                              color='black',alpha =1.)
p3

In [ ]:
(p2*p1*p3).opts(show_legend=False,width=700,height=250)